# Benchmark Gecode Base — résolution one-shot du CCMKP

Ce notebook résout le CCMKP via `gecode_cpp.solve_mkp` :
- Formulation CP avec **réification** (`z_i = 1 → A_i x ≤ b_i`, sans Big-M)
- Contrainte de chance `Σ z_i ≥ ⌈(1−ε)N⌉`
- Moteur B&B **depth-first** natif de Gecode (pas de borne Lagrangienne)

**Référence** pour comparer avec `bench_gecode_bab.ipynb` (V1LD/V1CLD).

In [1]:
# ── Paramètres ────────────────────────────────────────────────────────────────
DATASET     = 'data/knapsack/datasets-song/ccmk-song-mknap1_4_m5-S100.txt'
REPLICATION = 0
EPSILON     = 0.10
TIME_LIMIT  = 300.0   # limite de temps (secondes)
N_REPS      = 5       # nombre de réplications à parcourir

In [2]:
import os, sys, time
import numpy as np
import pandas as pd

try:
    ROOT = os.path.dirname(os.path.abspath(__vsc_ipynb_file__))
except NameError:
    ROOT = os.getcwd()

LIB = os.path.join(ROOT, 'lib')
for p in [LIB, ROOT]:
    if p not in sys.path:
        sys.path.insert(0, p)

import gecode_cpp as gc
print(f'ROOT   : {ROOT}')
print(f'Module : {gc.__file__}')

def load_song(filepath, replication=0):
    """Charge une réplication CCMKP Song. Retourne c, p, A (N,m,n), b (N,m)."""
    rows = []
    with open(filepath) as f:
        for line in f:
            v = list(map(float, line.strip().split(',')))
            if int(v[4]) != replication:
                continue
            m, n = int(v[2]), int(v[3])
            A = np.array(v[5 : 5+m*n]).reshape(m, n)
            b = np.array(v[5+m*n : 5+m*n+m])
            c = np.array(v[5+m*n+m : 5+m*n+m+n])
            rows.append((A, b, c))
    N = len(rows)
    assert N > 0, f'Aucun scénario pour replication={replication}'
    return (
        rows[0][2],
        np.ones(N) / N,
        np.stack([r[0] for r in rows]),   # (N, m, n)
        np.stack([r[1] for r in rows]),   # (N, m)
    )

c, p, A, b = load_song(os.path.join(ROOT, DATASET), REPLICATION)
N, m, n = A.shape
print(f'\nInstance : {os.path.basename(DATASET)}')
print(f'  n={n}, m={m}, N={N}, ε={EPSILON}, rep={REPLICATION}')

ROOT   : /Users/roman.lendormy/Documents/Cours/X25 - 3A/Stage/Learning_bounds_CCCP
Module : /Users/roman.lendormy/Documents/Cours/X25 - 3A/Stage/Learning_bounds_CCCP/lib/gecode_cpp.cpython-311-darwin.so

Instance : ccmk-song-mknap1_4_m5-S100.txt
  n=20, m=5, N=100, ε=0.1, rep=0


## 1. Résolution — réplication unique

In [3]:
print(f'Résolution rep={REPLICATION}...', flush=True)
t0 = time.perf_counter()
res = gc.solve_mkp(c, p, A, b, EPSILON, time_limit=TIME_LIMIT)
wall = time.perf_counter() - t0

print(f"\n  Statut           : {res['status']}")
print(f"  Valeur optimale  : {res['objective']:.4f}")
print(f"  Faisable         : {res['feasible']}")
print(f"  Optimal prouvé   : {res['optimal']}")
print(f"  Temps (C++)      : {res['solve_time_s']:.3f} s")
print(f"  Temps mur (nb)   : {wall:.3f} s")

Résolution rep=0...

  Statut           : OPTIMAL
  Valeur optimale  : 6070.0000
  Faisable         : True
  Optimal prouvé   : True
  Temps (C++)      : 0.705 s
  Temps mur (nb)   : 0.706 s


## 2. Affichage détaillé

In [4]:
def display_result(res, rep, n, m, N):
    t = res['solve_time_s']
    print(f'┌──────────────────────────────────────────────────┐')
    print(f'│  Gecode Base — one-shot CP (réification)         │')
    print(f'│  Instance : {os.path.basename(DATASET):<37}│'[:52] + '│')
    print(f'│  n={n}, m={m}, N={N}, ε={EPSILON}, rep={rep}                    │'[:52] + '│')
    print(f'│  time_limit={TIME_LIMIT:.0f}s                                  │'[:52] + '│')
    print(f'├──────────────────────────────────────────────────┤')
    print(f'│  Statut          : {res["status"]:<31}│')
    print(f'│  Valeur optimale : {res["objective"]:<31.4f}│')
    print(f'│  Optimal prouvé  : {str(res["optimal"]):<31}│')
    print(f'├──────────────────────────────────────────────────┤')
    print(f'│  Temps total (s) : {t:<31.3f}│')
    print(f'│  Nœuds DFS       : {res["nodes"]:<31}│')
    print(f'│  Échecs          : {res["fails"]:<31}│')
    print(f'└──────────────────────────────────────────────────┘')

display_result(res, REPLICATION, n, m, N)

┌──────────────────────────────────────────────────┐
│  Gecode Base — one-shot CP (réification)         │
│  Instance : ccmk-song-mknap1_4_m5-S100.txt       ││
│  n=20, m=5, N=100, ε=0.1, rep=0                   │
│  time_limit=300s                                  │
├──────────────────────────────────────────────────┤
│  Statut          : OPTIMAL                        │
│  Valeur optimale : 6070.0000                      │
│  Optimal prouvé  : True                           │
├──────────────────────────────────────────────────┤
│  Temps total (s) : 0.705                          │
│  Nœuds DFS       : 7954                           │
│  Échecs          : 3261                           │
└──────────────────────────────────────────────────┘


## 3. Sweep sur les 5 réplications

In [5]:
rows = []
for rep in range(N_REPS):
    c_r, p_r, A_r, b_r = load_song(os.path.join(ROOT, DATASET), rep)
    print(f'rep={rep}...', end=' ', flush=True)
    r = gc.solve_mkp(c_r, p_r, A_r, b_r, EPSILON, time_limit=TIME_LIMIT)
    rows.append({
        'Rep'            : rep,
        'Statut'         : r['status'],
        'Valeur opt.'    : r['objective'],
        'Optimal prouvé' : r['optimal'],
        'Temps (s)'      : f"{r['solve_time_s']:.3f}",
        'Nœuds DFS'      : r['nodes'],
        'Échecs'         : r['fails'],
    })
    print(f"obj={r['objective']:.1f}  t={r['solve_time_s']:.3f}s  nodes={r['nodes']}  [{r['status']}]")

df = pd.DataFrame(rows).set_index('Rep')
print(f'\n{df.to_string()}')
print(f'\nMoyenne temps  : {df["Temps (s)"].astype(float).mean():.3f} s')
print(f'Moy. nœuds DFS : {df["Nœuds DFS"].mean():.0f}')
print(f'Optimal prouvé : {df["Optimal prouvé"].sum()}/{N_REPS}')

rep=0... obj=6070.0  t=0.785s  nodes=7954  [OPTIMAL]
rep=1... obj=6050.0  t=0.790s  nodes=8572  [OPTIMAL]
rep=2... obj=6050.0  t=0.700s  nodes=7784  [OPTIMAL]
rep=3... obj=6010.0  t=0.708s  nodes=8453  [OPTIMAL]
rep=4... obj=6050.0  t=0.707s  nodes=7906  [OPTIMAL]

      Statut  Valeur opt.  Optimal prouvé Temps (s)  Nœuds DFS  Échecs
Rep                                                                   
0    OPTIMAL       6070.0            True     0.785       7954    3261
1    OPTIMAL       6050.0            True     0.790       8572    3572
2    OPTIMAL       6050.0            True     0.700       7784    3314
3    OPTIMAL       6010.0            True     0.708       8453    3509
4    OPTIMAL       6050.0            True     0.707       7906    3300

Moyenne temps  : 0.738 s
Moy. nœuds DFS : 8134
Optimal prouvé : 5/5
